# Setup and Settings

In [ ]:
from collections.abc import Iterable
from datetime import date, datetime, time, timedelta
from decimal import Decimal
from math import isfinite
from pathlib import Path
from sys import float_info
from typing import Any, cast

import pandas as pd
import pyarrow.parquet as pq  # pyright: ignore[reportMissingTypeStubs]
from openpyxl import Workbook
from openpyxl.worksheet.worksheet import Worksheet

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)

source_path = project_root / r"data/processed/file.parquet"

# Load and Preview

In [ ]:
# PyArrow does not publish types for its Parquet table reader.
table = cast(Any, pq.read_table(source_path))  # pyright: ignore[reportUnknownMemberType]
schema = table.schema
df = pd.DataFrame(table.to_pandas(types_mapper=pd.ArrowDtype))
df

# Prepare Excel Workbook

In [ ]:
def excel_value(value: object) -> str | int | float | bool | None:
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, str):
        if len(value.encode("utf-16-le")) // 2 > 32_767:
            raise ValueError(
                "A text value exceeds Excel's 32,767-character cell limit."
            )
        return value
    if isinstance(value, bool):
        return value
    if isinstance(value, int):
        return str(value) if len(str(abs(value))) > 15 else value
    if isinstance(value, float):
        if not isfinite(value):
            return str(value)
        if 0.0 < abs(value) < float_info.min:
            return str(value)
        return str(value) if len(Decimal(str(value)).as_tuple().digits) > 15 else value
    if isinstance(value, Decimal):
        return str(value)
    if isinstance(value, (datetime, date, time)):
        return value.isoformat()
    if isinstance(value, timedelta):
        return str(value)
    raise ValueError(f"Unsupported Excel value type: {type(value).__name__}")


def write_row(sheet: Worksheet, row_number: int, values: Iterable[object]) -> None:
    for column_number, value in enumerate(values, start=1):
        converted = excel_value(value)
        cell = sheet.cell(row=row_number, column=column_number, value=converted)
        if isinstance(converted, str):
            # openpyxl otherwise interprets formulas and Excel error strings.
            cell.data_type = "s"


if len(df) + 1 > 1_048_576 or len(df.columns) > 16_384:
    raise ValueError("The data exceeds Excel's sheet limits, including the header.")
if not len(df.columns):
    raise ValueError("The source has no columns to export.")

workbook = Workbook()
data_sheet = workbook.active
if data_sheet is None:
    raise RuntimeError("The workbook has no active sheet.")
data_sheet.title = "data"
write_row(data_sheet, 1, df.columns)
for row_number, row in enumerate(df.itertuples(index=False, name=None), start=2):
    write_row(data_sheet, row_number, row)
data_sheet.freeze_panes = "A2"
data_sheet.auto_filter.ref = data_sheet.dimensions

schema_sheet = workbook.create_sheet("data_types")
write_row(schema_sheet, 1, ["column_name", "parquet_type"])
for row_number, field in enumerate(schema, start=2):
    write_row(schema_sheet, row_number, [field.name, str(field.type)])
schema_sheet.freeze_panes = "A2"
schema_sheet.auto_filter.ref = schema_sheet.dimensions

# Save Excel Export

In [ ]:
output_dir = project_root / "data/exports"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{source_path.stem}.xlsx"
with output_path.open("xb") as output_file:
    workbook.save(output_file)